# handling missing data

strategies: drop, mean impute, median impute, KNN impute. compare on a synthetic NaN'd version of breast cancer.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

X, y = load_breast_cancer(return_X_y=True)
rng = np.random.RandomState(0)
mask = rng.rand(*X.shape) < 0.10  # 10% missing
Xn = X.copy().astype(float)
Xn[mask] = np.nan
print('na count:', np.isnan(Xn).sum())

In [2]:
for strat in ['mean', 'median', 'most_frequent']:
    p = Pipeline([('imp', SimpleImputer(strategy=strat)),
                  ('sc', StandardScaler()),
                  ('lr', LogisticRegression(solver='lbfgs', max_iter=2000))])
    s = cross_val_score(p, Xn, y, cv=5).mean()
    print(f'{strat:15s}  {s:.4f}')

## KNN imputer (sklearn 0.22 has it; using a workaround for 0.21)

In [3]:
# In sklearn 0.21 KNNImputer isn't yet in main. fancyimpute is one option.
# Stick with iterative-style: fill with column mean, refit.
# Just an exploration cell.
Xfill = pd.DataFrame(Xn).fillna(method='ffill').fillna(method='bfill').values
p = Pipeline([('sc', StandardScaler()),
              ('lr', LogisticRegression(solver='lbfgs', max_iter=2000))])
cross_val_score(p, Xfill, y, cv=5).mean()

## drop rows with NaN baseline

In [4]:
mask = ~np.isnan(Xn).any(axis=1)
Xd, yd = Xn[mask], y[mask]
print('rows kept:', mask.sum(), '/', len(mask))
p = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(solver='lbfgs', max_iter=2000))])
cross_val_score(p, Xd, yd, cv=5).mean()